In [1]:
# ==============================================================================
# Cell 1: Ingesta de la Realidad Absoluta (v_ML_Supervised) y el GAN
# Objetivo: Descargar la tabla madre que contiene los IDs y coordenadas crudas.
# ==============================================================================
import pandas as pd
import numpy as np
import geopandas as gpd
from google.colab import auth
from google.cloud import bigquery

print("🚀 INICIANDO DOWNSCALING (MÉTODO DE PARIDAD NATIVA)")
print("-" * 65)

auth.authenticate_user()
PROJECT_ID = 'drivers-dilemma'
client = bigquery.Client(project=PROJECT_ID)

# 1. INGESTA SINTÉTICA (GAN) - MODIFICADO PARA TRAER TODAS LAS COLUMNAS
print("⏳ Descargando Manifold Sintético Completo...")
query_gan = f"""
    SELECT *
    FROM `{PROJECT_ID}.pienza_big.synthetic_manifold_v8_enriched`
"""
df_synthetic = client.query(query_gan).to_dataframe()
print(f"   ✅ GAN descargado: {len(df_synthetic):,} viajes con {len(df_synthetic.columns)} columnas.")

# 2. INGESTA DE REALIDAD (Con IDs Crudos)
print("⏳ Descargando Realidad Absoluta (v_ML_Supervised)...")
query_real = f"""
    SELECT
        pickup_lat, pickup_lon,
        dropoff_polygon_id, dropoff_polygon_name,
        dropoff_hdbscan_id, dropoff_hdbscan_name
    FROM `{PROJECT_ID}.pienza_mini.v_ML_Supervised`
"""
df_real = client.query(query_real).to_dataframe()

# Sanitización de tipos para emular el preprocesamiento del GAN
df_real['dropoff_polygon_id'] = pd.to_numeric(df_real['dropoff_polygon_id'], errors='coerce')
df_real['dropoff_hdbscan_id'] = pd.to_numeric(df_real['dropoff_hdbscan_id'], errors='coerce')

print(f"   ✅ Realidad descargada: {len(df_real):,} registros base.")
print("-" * 65)

🚀 INICIANDO DOWNSCALING (MÉTODO DE PARIDAD NATIVA)
-----------------------------------------------------------------
⏳ Descargando Manifold Sintético Completo...
   ✅ GAN descargado: 1,010,001 viajes con 14 columnas.
⏳ Descargando Realidad Absoluta (v_ML_Supervised)...
   ✅ Realidad descargada: 4,765 registros base.
-----------------------------------------------------------------


In [2]:
# ==============================================================================
# Cell 2: Reconstrucción Clónica de IDs (Macro) y Nombres (Micro) - FIX APLICADO
# Objetivo: Replicar el código exacto del GAN asegurando paridad ID-Nombre.
# ==============================================================================
import pandas as pd
import numpy as np
import geopandas as gpd

print("🧮 RECONSTRUYENDO IDs Y NOMBRES DESDE LA BASE (PATCH: REFORMA SOCIAL)...")
print("-" * 65)

id_map = {
    -1:-1, 41:0, 42:0, 46:0, 43:1, 65:2, 62:2, 44:2, 36:2, 49:3, 52:3, 35:3, 50:4, 58:4,
    25:5, 31:5, 63:6, 39:6, 51:7, 33:7, 37:8, 53:8, 48:8, 60:9, 57:10, 12:10, 32:10,
    24:11, 40:12, 45:13, 59:13, 61:14, 38:14, 34:15, 30:16, 66:16, 17:17, 14:17, 22:17,
    16:18, 13:18, 11:19, 15:20, 21:21, 20:21, 19:21, 18:22, 47:23, 55:23, 56:23, 54:24,
    64:24, 71:25, 9:26, 70:27, 69:28, 8:29, 6:30, 7:30, 23:30, 3:31, 2:32, 4:33, 29:33,
    68:34, 5:35, 27:36, 28:36, 1:37, 10:38, 0:39, 26:40, 67:41
}

# --- 1. PROCESAMIENTO DE DESTINOS (DROPOFFS) ---
print("✅ Procesando Destinos (Replicando Lógica del GAN)...")

# [PATCH] Desambiguación de Nombres en Dropoffs vía ID Único
df_real.loc[df_real['dropoff_polygon_id'] == 19, 'dropoff_polygon_name'] = 'reforma_social'
df_real.loc[df_real['dropoff_polygon_id'] == 66, 'dropoff_polygon_name'] = 'tecamachalco'

df_real['id_agrupado_drop'] = df_real['dropoff_polygon_id'].fillna(-1).astype(int).map(id_map).fillna(-1).astype(int)

# Las reglas de oro de tu Cell 6.10
cond_drop = [(df_real['id_agrupado_drop'] >= 0), (df_real['dropoff_hdbscan_id'] > -1)]

# A. MACRO ID
choice_id = ["P_" + df_real['id_agrupado_drop'].astype(str), "C_" + df_real['dropoff_hdbscan_id'].fillna(-1).astype(int).astype(str)]
df_real['dropoff_zone_id'] = np.select(cond_drop, choice_id, default="Unassigned")

# B. MICRO NAME
choice_name = [df_real['dropoff_polygon_name'], df_real['dropoff_hdbscan_name']]
df_real['dropoff_micro_name'] = np.select(cond_drop, choice_name, default="unassigned_area")
df_real['dropoff_micro_name'] = df_real['dropoff_micro_name'].astype(str).str.strip().str.lower()

# C. Agrupación
df_mini_dropoffs = df_real.groupby(['dropoff_zone_id', 'dropoff_micro_name']).size().reset_index(name='hist_dropoffs')

# --- 2. PROCESAMIENTO DE ORÍGENES (PICKUPS) ---
print("✅ Procesando Orígenes (Replicando Spatial Join)...")
df_pickups_clean = df_real.dropna(subset=['pickup_lat', 'pickup_lon']).copy()
gdf_pickups = gpd.GeoDataFrame(df_pickups_clean, geometry=gpd.points_from_xy(df_pickups_clean.pickup_lon, df_pickups_clean.pickup_lat), crs="EPSG:4326")
gdf_polygons = gpd.read_file("/content/drive/MyDrive/_Pienza/Assets/Phase_2/poly.geojson").to_crs("EPSG:4326")

# [PATCH] Desambiguación Topológica en Pickups (El caso Tecamachalco)
tecas = gdf_polygons[gdf_polygons['name'].str.lower().str.strip() == 'tecamachalco']

if len(tecas) > 1:
    idx_este = tecas.geometry.centroid.x.idxmax()
    idx_oeste = tecas.geometry.centroid.x.idxmin()
    gdf_polygons.at[idx_este, 'name'] = 'reforma_social'
    gdf_polygons.at[idx_oeste, 'name'] = 'tecamachalco'
    print("   -> Corrección aplicada: 'reforma_social' (Este) y 'tecamachalco' (Oeste) separados.")

joined = gpd.sjoin(gdf_pickups, gdf_polygons[['geometry', 'name']], how="left", predicate='within')
joined = joined[~joined.index.duplicated(keep='first')]

# A. MACRO ID
joined['pickup_agrupado'] = joined['index_right'].map(id_map).fillna(-1).astype(int)
joined['pickup_zone_id'] = np.where(joined['pickup_agrupado'] >= 0, "P_" + joined['pickup_agrupado'].astype(str), "Unassigned")

# B. MICRO NAME
joined['pickup_micro_name'] = joined['name'].fillna('unassigned_area').astype(str).str.strip().str.lower()

# C. Agrupación
df_mini_pickups = joined.groupby(['pickup_zone_id', 'pickup_micro_name']).size().reset_index(name='hist_pickups')

print("-" * 65)
print(f"🏆 CIMENTACIÓN LISTA:")
print(f"   - Total Pickups Históricos: {df_mini_pickups['hist_pickups'].sum():,.0f}")
print(f"   - Total Dropoffs Históricos: {df_mini_dropoffs['hist_dropoffs'].sum():,.0f}")

🧮 RECONSTRUYENDO IDs Y NOMBRES DESDE LA BASE (PATCH: REFORMA SOCIAL)...
-----------------------------------------------------------------
✅ Procesando Destinos (Replicando Lógica del GAN)...
✅ Procesando Orígenes (Replicando Spatial Join)...
   -> Corrección aplicada: 'reforma_social' (Este) y 'tecamachalco' (Oeste) separados.
-----------------------------------------------------------------
🏆 CIMENTACIÓN LISTA:
   - Total Pickups Históricos: 4,762
   - Total Dropoffs Históricos: 4,765


/tmp/ipykernel_3278/3249363382.py:55: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  idx_este = tecas.geometry.centroid.x.idxmax()
/tmp/ipykernel_3278/3249363382.py:56: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  idx_oeste = tecas.geometry.centroid.x.idxmin()


In [3]:
# ==============================================================================
# Cell 3: Motor de Downscaling (Outer Join + Redondeo Blindado)
# ==============================================================================
import pandas as pd
import numpy as np

print("⚖️ INICIANDO DOWNSCALING MATEMÁTICO...")
print("-" * 65)

# --- 1. PICKUPS ---
df_mini_pickups['macro_total'] = df_mini_pickups.groupby('pickup_zone_id')['hist_pickups'].transform('sum')
df_mini_pickups['weight'] = df_mini_pickups['hist_pickups'] / df_mini_pickups['macro_total']

gan_p = df_synthetic.groupby('pickup_zone_id').size().reset_index(name='gan_pickups')
TARGET_P = gan_p['gan_pickups'].sum() # <--- EL TOTAL REAL (Antes del Merge)

downscale_p = pd.merge(df_mini_pickups, gan_p, on='pickup_zone_id', how='outer')
downscale_p['weight'] = downscale_p['weight'].fillna(1.0)
downscale_p['pickup_micro_name'] = downscale_p['pickup_micro_name'].fillna(downscale_p['pickup_zone_id'])
downscale_p['gan_pickups'] = downscale_p['gan_pickups'].fillna(0)

# Multiplicación y Redondeo
downscale_p['micro_gan_pickups'] = (downscale_p['gan_pickups'] * downscale_p['weight'])
downscale_p['micro_gan_pickups'] = downscale_p['micro_gan_pickups'].round(0).fillna(0).astype(int)

# Ajuste de diferencias por redondeo
diff_p = int(TARGET_P - downscale_p['micro_gan_pickups'].sum())
if diff_p != 0:
    max_idx = downscale_p['micro_gan_pickups'].idxmax()
    downscale_p.loc[max_idx, 'micro_gan_pickups'] += diff_p

# --- 2. DROPOFFS ---
df_mini_dropoffs['macro_total'] = df_mini_dropoffs.groupby('dropoff_zone_id')['hist_dropoffs'].transform('sum')
df_mini_dropoffs['weight'] = df_mini_dropoffs['hist_dropoffs'] / df_mini_dropoffs['macro_total']

gan_d = df_synthetic.groupby('dropoff_zone_id').size().reset_index(name='gan_dropoffs')
TARGET_D = gan_d['gan_dropoffs'].sum() # <--- EL TOTAL REAL (Antes del Merge)

downscale_d = pd.merge(df_mini_dropoffs, gan_d, on='dropoff_zone_id', how='outer')
downscale_d['weight'] = downscale_d['weight'].fillna(1.0)
downscale_d['dropoff_micro_name'] = downscale_d['dropoff_micro_name'].fillna(downscale_d['dropoff_zone_id'])
downscale_d['gan_dropoffs'] = downscale_d['gan_dropoffs'].fillna(0)

# Multiplicación y Redondeo
downscale_d['micro_gan_dropoffs'] = (downscale_d['gan_dropoffs'] * downscale_d['weight'])
downscale_d['micro_gan_dropoffs'] = downscale_d['micro_gan_dropoffs'].round(0).fillna(0).astype(int)

# Ajuste de diferencias por redondeo
diff_d = int(TARGET_D - downscale_d['micro_gan_dropoffs'].sum())
if diff_d != 0:
    max_idx = downscale_d['micro_gan_dropoffs'].idxmax()
    downscale_d.loc[max_idx, 'micro_gan_dropoffs'] += diff_d

print(f"🏆 VOLUMEN REPARTIDO Y REDONDEADO:")
print(f"   -> Pickups:  {downscale_p['micro_gan_pickups'].sum():,.0f} (Target: {TARGET_P:,.0f})")
print(f"   -> Dropoffs: {downscale_d['micro_gan_dropoffs'].sum():,.0f} (Target: {TARGET_D:,.0f})")

⚖️ INICIANDO DOWNSCALING MATEMÁTICO...
-----------------------------------------------------------------
🏆 VOLUMEN REPARTIDO Y REDONDEADO:
   -> Pickups:  1,010,001 (Target: 1,010,001)
   -> Dropoffs: 1,010,001 (Target: 1,010,001)


In [4]:
# ==============================================================================
# Cell 4: Auditoría de Fidelidad 1 vs 1
# ==============================================================================
def format_audit(df, name_col, hist_col, gan_col):
    audit = df[[name_col, hist_col, gan_col]].copy()

    # Agrupamos por Micro Name (Por si el outer join duplicó algo)
    audit = audit.groupby(name_col).sum().reset_index()

    t_hist = audit[hist_col].sum()
    t_gan = audit[gan_col].sum()

    audit['share_real'] = (audit[hist_col] / t_hist) * 100 if t_hist > 0 else 0
    audit['share_synth'] = (audit[gan_col] / t_gan) * 100 if t_gan > 0 else 0
    audit['delta_abs'] = audit['share_synth'] - audit['share_real']

    mae = audit['delta_abs'].abs().mean()

    audit['share_real'] = audit['share_real'].map('{:.4f} %'.format)
    audit['share_synth'] = audit['share_synth'].map('{:.4f} %'.format)
    audit['delta_abs'] = audit['delta_abs'].map('{:+.4f} %'.format)

    return audit.sort_values(by=name_col).reset_index(drop=True), mae

audit_p_df, mae_p = format_audit(downscale_p, 'pickup_micro_name', 'hist_pickups', 'micro_gan_pickups')
audit_d_df, mae_d = format_audit(downscale_d, 'dropoff_micro_name', 'hist_dropoffs', 'micro_gan_dropoffs')

print("📋 ESCÁNER DE FIDELIDAD 1 VS 1 (SHARE %)")
print("-" * 85)
print(f"📍 AUDITORÍA DE ORÍGENES (PICKUPS) | MAE: {mae_p:.6f} %")
with pd.option_context('display.max_rows', 100): display(audit_p_df)

print("-" * 85)
print(f"🏁 AUDITORÍA DE DESTINOS (DROPOFFS) | MAE: {mae_d:.6f} %")
with pd.option_context('display.max_rows', 100): display(audit_d_df)

📋 ESCÁNER DE FIDELIDAD 1 VS 1 (SHARE %)
-------------------------------------------------------------------------------------
📍 AUDITORÍA DE ORÍGENES (PICKUPS) | MAE: 0.124343 %


,pickup_micro_name,hist_pickups,micro_gan_pickups,share_real,share_synth,delta_abs
0,agwa_bezares,30,5675,0.6300 %,0.5619 %,-0.0681 %
1,ahuehuetes_norte,36,6765,0.7560 %,0.6698 %,-0.0862 %
2,ahuehuetes_sur,71,16307,1.4910 %,1.6146 %,+0.1236 %
3,anahuac_1,24,4915,0.5040 %,0.4866 %,-0.0174 %
4,anzures,303,69333,6.3629 %,6.8646 %,+0.5018 %
5,ave_club_de_golf_lomas,2,492,0.0420 %,0.0487 %,+0.0067 %
6,bahias,85,17408,1.7850 %,1.7236 %,-0.0614 %
7,blvrd_anahuac,19,3610,0.3990 %,0.3574 %,-0.0416 %
8,bondojito_asf,6,1196,0.1260 %,0.1184 %,-0.0076 %
9,bosque_2,10,1992,0.2100 %,0.1972 %,-0.0128 %


-------------------------------------------------------------------------------------
🏁 AUDITORÍA DE DESTINOS (DROPOFFS) | MAE: 0.118574 %


,dropoff_micro_name,hist_dropoffs,micro_gan_dropoffs,share_real,share_synth,delta_abs
0,agwa_bezares,19,3610,0.3987 %,0.3574 %,-0.0413 %
1,ahuehuetes_norte,19,3556,0.3987 %,0.3521 %,-0.0467 %
2,ahuehuetes_sur,48,11866,1.0073 %,1.1749 %,+0.1675 %
3,anahuac_1,24,5715,0.5037 %,0.5658 %,+0.0622 %
4,anzures,48,9397,1.0073 %,0.9304 %,-0.0770 %
5,ave_club_de_golf_lomas,3,542,0.0630 %,0.0537 %,-0.0093 %
6,bahias,15,3572,0.3148 %,0.3537 %,+0.0389 %
7,barranca_del_muerto,32,6064,0.6716 %,0.6004 %,-0.0712 %
8,blvrd_anahuac,12,3204,0.2518 %,0.3172 %,+0.0654 %
9,bondojito_asf,13,3371,0.2728 %,0.3338 %,+0.0609 %


In [5]:
# ==============================================================================
# Cell 5: Inyección de Nombres Micro, Renombrado y Persistencia Final
# Objetivo: Asignar nombres reales, clarificar nomenclatura y guardar Parquet.
# ==============================================================================
import os
import pandas as pd

print("📦 FORJANDO EL MANIFOLD DEFINITIVO (DOWN-MAPPED)...")
print("-" * 65)

# --- 1. CREACIÓN DE LOS POOLS DE IDENTIDAD ---
print("🎲 Generando pools de nombres estocásticos...")

pickup_pool = downscale_p.loc[downscale_p.index.repeat(downscale_p['micro_gan_pickups'])].copy()
pickup_pool = pickup_pool[['pickup_zone_id', 'pickup_micro_name']].sample(frac=1).reset_index(drop=True)

dropoff_pool = downscale_d.loc[downscale_d.index.repeat(downscale_d['micro_gan_dropoffs'])].copy()
dropoff_pool = dropoff_pool[['dropoff_zone_id', 'dropoff_micro_name']].sample(frac=1).reset_index(drop=True)

# --- 2. MAPEO POR REEMPLAZO SECUENCIAL ---
print("💉 Inyectando identidades micro al millón de viajes...")

# Al ordenar, pandas moverá TODAS las columnas originales juntas
df_synthetic = df_synthetic.sort_values('pickup_zone_id').reset_index(drop=True)
pickup_pool = pickup_pool.sort_values('pickup_zone_id').reset_index(drop=True)
df_synthetic['pickup_micro_name'] = pickup_pool['pickup_micro_name'].values

df_synthetic = df_synthetic.sort_values('dropoff_zone_id').reset_index(drop=True)
dropoff_pool = dropoff_pool.sort_values('dropoff_zone_id').reset_index(drop=True)
df_synthetic['dropoff_micro_name'] = dropoff_pool['dropoff_micro_name'].values

# --- 3. MEZCLA FINAL ---
# Mezclamos para romper el orden de los IDs
df_synthetic = df_synthetic.sample(frac=1).reset_index(drop=True)

# --- 4. RENOMBRADO Y REORDENAMIENTO ---
print("🏷️  Aplicando nueva nomenclatura (GAN vs Downscaled)...")

rename_dict = {
    'dropoff_zone_id': 'dropoff_id_GAN',
    'pickup_zone_id': 'pickup_id_GAN',
    'dropoff_name': 'dropoff_name_GAN',
    'pickup_name': 'pickup_name_GAN',
    'pickup_micro_name': 'pickup_name_down',
    'dropoff_micro_name': 'dropoff_name_down'
}
df_synthetic = df_synthetic.rename(columns=rename_dict)

# Reordenamos exactamente a tu gusto
col_order = [
    'upfront_fare', 'est_trip_time_sec', 'est_trip_dist_km',
    'time_to_pickup_sec', 'dist_to_pickup_km', 'hour_of_day',
    'day_of_week', 'product_category_fk', 'dropoff_id_GAN',
    'pickup_id_GAN', 'reason_primary_fk', 'product_name',
    'dropoff_name_GAN', 'pickup_name_GAN', 'pickup_name_down',
    'dropoff_name_down'
]

# Protegemos el código por si en el futuro agregas más columnas a la query base
existing_cols = [c for c in col_order if c in df_synthetic.columns]
remaining_cols = [c for c in df_synthetic.columns if c not in existing_cols]
df_synthetic = df_synthetic[existing_cols + remaining_cols]

print("✅ Inyección y renombrado completados exitosamente.")

# --- 5. AUDITORÍA ---
print("\n🔍 AUDITORÍA DE COLUMNAS EN EL DATAFRAME FINAL:")
print("-" * 40)
for i, col in enumerate(df_synthetic.columns, 1):
    print(f" {i}. {col}")
print("-" * 40)

# --- 6. GUARDADO EN DISCO ---
SAVE_PATH = '/content/drive/MyDrive/_Pienza/Assets/Phase_6/synthetic_manifold_v8_downscaled.parquet'
os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)

print(f"\n💾 Guardando Parquet definitivo en: {SAVE_PATH}")
df_synthetic.to_parquet(SAVE_PATH, index=False)

# Validación de tamaño
file_size = os.path.getsize(SAVE_PATH) / (1024 * 1024)
print(f"✅ ARCHIVO ASEGURADO: {file_size:.2f} MB")
print("-" * 65)
print("🏁 EL MOTOR DE DOWNSCALING HA TERMINADO. LA ARENA ESTÁ LISTA.")

📦 FORJANDO EL MANIFOLD DEFINITIVO (DOWN-MAPPED)...
-----------------------------------------------------------------
🎲 Generando pools de nombres estocásticos...
💉 Inyectando identidades micro al millón de viajes...
🏷️  Aplicando nueva nomenclatura (GAN vs Downscaled)...
✅ Inyección y renombrado completados exitosamente.

🔍 AUDITORÍA DE COLUMNAS EN EL DATAFRAME FINAL:
----------------------------------------
 1. upfront_fare
 2. est_trip_time_sec
 3. est_trip_dist_km
 4. time_to_pickup_sec
 5. dist_to_pickup_km
 6. hour_of_day
 7. day_of_week
 8. product_category_fk
 9. dropoff_id_GAN
 10. pickup_id_GAN
 11. reason_primary_fk
 12. product_name
 13. dropoff_name_GAN
 14. pickup_name_GAN
 15. pickup_name_down
 16. dropoff_name_down
----------------------------------------

💾 Guardando Parquet definitivo en: /content/drive/MyDrive/_Pienza/Assets/Phase_6/synthetic_manifold_v8_downscaled.parquet
✅ ARCHIVO ASEGURADO: 36.68 MB
-----------------------------------------------------------------


In [6]:
# ==============================================================================
# Cell 1.6: Auditoría Post-Desambiguación (Estado en Memoria)
# Objetivo: Validar que 'reforma_social' y 'tecamachalco' coexistan en gdf_polygons
# ==============================================================================

print("🔍 AUDITANDO RESULTADO DE LA DESAMBIGUACIÓN (IN-MEMORY)")
print("-" * 65)

# 1. Verificación directa de existencia de nombres
target_names = ['reforma_social', 'tecamachalco']
found_names = gdf_polygons[gdf_polygons['name'].isin(target_names)]

print(f"📊 Reporte de Polígonos Críticos:")
if not found_names.empty:
    for name in target_names:
        count = len(gdf_polygons[gdf_polygons['name'] == name])
        print(f"   -> '{name}': {count} polígono(s) encontrado(s).")
else:
    print("   ⚠️ Error: No se encontraron los nombres esperados.")

print("-" * 65)

# 2. Análisis de Duplicados General
name_counts_final = gdf_polygons['name'].value_counts()
dupes_final = name_counts_final[name_counts_final > 1]

if dupes_final.empty:
    print("✅ LIMPIEZA TOTAL: No existen nombres duplicados en gdf_polygons.")
else:
    print(f"⚠️ Atención: Aún persisten duplicados en: {dupes_final.index.tolist()}")

# 3. Muestra de Coordenadas Finales
if len(found_names) >= 2:
    print("\n📍 Coordenadas Finales (Verificación de Eje X):")
    for idx, row in found_names.iterrows():
        print(f"   - {row['name'].ljust(15)} | Longitud (X): {row.geometry.centroid.x:.5f}")

print("-" * 65)

🔍 AUDITANDO RESULTADO DE LA DESAMBIGUACIÓN (IN-MEMORY)
-----------------------------------------------------------------
📊 Reporte de Polígonos Críticos:
   -> 'reforma_social': 1 polígono(s) encontrado(s).
   -> 'tecamachalco': 1 polígono(s) encontrado(s).
-----------------------------------------------------------------
✅ LIMPIEZA TOTAL: No existen nombres duplicados en gdf_polygons.

📍 Coordenadas Finales (Verificación de Eje X):
   - reforma_social  | Longitud (X): -99.22069
   - tecamachalco    | Longitud (X): -99.22957
-----------------------------------------------------------------


In [7]:
# ==============================================================================
# Cell 6: Auditoría Específica - El Duelo: Reforma Social vs. Tecamachalco
# Objetivo: Validar paridad y realismo en el reparto de los nodos desambiguados.
# ==============================================================================

print("🎯 AUDITORÍA DE PRECISIÓN: REFORMA SOCIAL VS. TECAMACHALCO")
print("-" * 85)

targets = ['reforma_social', 'tecamachalco']

# --- 1. AUDITORÍA PICKUPS (Basado en el SJOIN desambiguado) ---
p_audit = downscale_p[downscale_p['pickup_micro_name'].isin(targets)].copy()
p_audit['share_in_group'] = (p_audit['micro_gan_pickups'] / p_audit['micro_gan_pickups'].sum()) * 100

print("📍 ORIGENES (PICKUPS) - Basado en Geometría Real:")
if not p_audit.empty:
    display(p_audit[['pickup_micro_name', 'hist_pickups', 'micro_gan_pickups', 'share_in_group']]
            .sort_values('pickup_micro_name'))
else:
    print("⚠️ No se encontraron datos para estos nodos en Pickups.")

print("\n" + "-" * 85)

# --- 2. AUDITORÍA DROPOFFS (Basado en ID_Map + Nomenclatura Real) ---
d_audit = downscale_d[downscale_d['dropoff_micro_name'].isin(targets)].copy()
d_audit['share_in_group'] = (d_audit['micro_gan_dropoffs'] / d_audit['micro_gan_dropoffs'].sum()) * 100

print("🏁 DESTINOS (DROPOFFS) - Basado en Etiquetado Histórico:")
if not d_audit.empty:
    display(d_audit[['dropoff_micro_name', 'hist_dropoffs', 'micro_gan_dropoffs', 'share_in_group']]
            .sort_values('dropoff_micro_name'))
else:
    print("⚠️ No se encontraron datos para estos nodos en Dropoffs.")

print("-" * 85)
print("💡 NOTA: Si en Dropoffs 'reforma_social' aparece en 0, es porque el log histórico")
print("   de Uber (v_ML_Supervised) los etiqueta a ambos como 'tecamachalco'.")

🎯 AUDITORÍA DE PRECISIÓN: REFORMA SOCIAL VS. TECAMACHALCO
-------------------------------------------------------------------------------------
📍 ORIGENES (PICKUPS) - Basado en Geometría Real:


,pickup_micro_name,hist_pickups,micro_gan_pickups,share_in_group
26,reforma_social,13,2493,43.333913
14,tecamachalco,18,3260,56.666087



-------------------------------------------------------------------------------------
🏁 DESTINOS (DROPOFFS) - Basado en Etiquetado Histórico:


,dropoff_micro_name,hist_dropoffs,micro_gan_dropoffs,share_in_group
52,reforma_social,8,2223,62.496486
39,tecamachalco,7,1334,37.503514


-------------------------------------------------------------------------------------
💡 NOTA: Si en Dropoffs 'reforma_social' aparece en 0, es porque el log histórico
   de Uber (v_ML_Supervised) los etiqueta a ambos como 'tecamachalco'.


In [ ]:
from google.cloud import bigquery

# 1. Initialize Client
PROJECT_ID = "drivers-dilemma"
client = bigquery.Client(project=PROJECT_ID)

# 2. Configuration (UPDATE YOUR BUCKET NAME HERE)
GCS_BUCKET = "pienza-streamlit"
FILE_NAME = "260501_synthetic_manifold_v8_downscaled.parquet"
GCS_URI = f"gs://{GCS_BUCKET}/{FILE_NAME}"

# 3. The DDL Query
query_create_external = f"""
CREATE OR REPLACE EXTERNAL TABLE `{PROJECT_ID}.pienza_big.synthetic_manifold_v8_downscaled`
OPTIONS (
  format = 'PARQUET',
  uris = ['{GCS_URI}']
);
"""

print(f"🔗 Linking BigQuery to GCS at: {GCS_URI}...")

try:
    # Execute the query
    query_job = client.query(query_create_external)
    query_job.result()  # Wait for completion

    print("\n🚀 MISSION ACCOMPLISHED!")
    print(f"External table successfully created: {PROJECT_ID}.pienza_big.synthetic_manifold_v8_downscaled")
    print("Your Streamlit dashboard is now cleared for takeoff.")

except Exception as e:
    print("\n❌ CRITICAL ERROR:")
    print(e)